In [1]:
from gensim.models.fasttext import load_facebook_vectors
from numpy import dot
from numpy.linalg import norm

print("Loading FastText vectors...")

ft = load_facebook_vectors("models/cc.en.300.bin")  # needs ~4–5 GB RAM

print("FastText vectors loaded.")

def cos_sim(w1, w2):
    v1, v2 = ft.get_vector(w1), ft.get_vector(w2)  # works for rare/OOV
    return float(dot(v1, v2) / (norm(v1) * norm(v2)))

Loading FastText vectors...
FastText vectors loaded.


In [2]:
print(cos_sim("king", "queen"))  # example usage
print(cos_sim("apple", "banana"))
print(cos_sim("computer", "banana"))

0.7068519592285156
0.5262576341629028
0.14826184511184692


In [5]:
import pandas as pd
phonosemantic_radicals = pd.read_csv("phonosemantic_radicals.csv")



def get_most_similar_radical(word):
    word_vector = ft.get_vector(word.lower())
    radicals = phonosemantic_radicals.to_dict(orient='records')
    similarities = []
    for radical in radicals:
        word = radical['Standalone']
        meaning = radical['Meaning']
        seed_words = radical['Seed Words'].split()
        words = [meaning] + seed_words
        similarity_scores = []
        for w in words:
            try:
                w_vector = ft.get_vector(w.lower())
                similarity = dot(word_vector, w_vector) / (norm(word_vector) * norm(w_vector))
                similarity_scores.append((w, similarity))
            except KeyError:
                continue
        if not similarity_scores:
            continue
        top_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)[:3]
        avg_similarity = sum(score for _, score in top_scores) / len(top_scores)
        max_similarity_words = [w for w, s in top_scores]
        similarities.append((word, avg_similarity, ", ".join(max_similarity_words)))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[0]


In [36]:
from phonetic_similarity import get_best_phonetic_components

def get_character(word : str, phones : str | None = None):
    semantic_component = get_most_similar_radical(word)
    phonetic_components = get_best_phonetic_components(phones if phones else word.lower())
    replaced_phonetic_components = []
    for score, phone_word, phone_glyph in phonetic_components:
        word_vector = ft.get_vector(word.lower())
        phone_vector = ft.get_vector(phone_word.lower())
        similarity = dot(word_vector, phone_vector) / (norm(word_vector) * norm(phone_vector))
        replaced_phonetic_components.append((score + similarity, phone_word, phone_glyph))
    glyph_parts = (semantic_component[0], replaced_phonetic_components[0][2])
    word_parts = (word, replaced_phonetic_components[0][1])
    return (glyph_parts, word_parts)


In [10]:
with open("word_morph_phones.txt", "r", encoding="utf-8") as f:
    word_morph_phones = {}
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) < 3:
            continue
        word = parts[0]
        morphemes = parts[1].split("|")
        phones = parts[2].split("|")
        word_morph_phones[word] = (morphemes, phones)
        
print(word_morph_phones["necessity"])

(['necess', 'ity'], ['n @ s * e s', '@ t iy'])


In [54]:
import json
    
def get_word_glyphs(word: str) -> list[str]:
    with open("dictionary.json", "r", encoding="utf-8") as f:
        dictionary = json.load(f)
    if word in dictionary:
        return dictionary[word].split()
    
    glyphs = []
    if word in word_morph_phones:
        morphemes, phones = word_morph_phones[word]
        for morpheme, phone in zip(morphemes, phones):
            if morpheme in dictionary:
                glyphs.extend(dictionary[morpheme].split())
            else:
                character = get_character(morpheme, phone.split())
                glyph = "⿰" + character[0][0] + character[0][1]
                glyphs.append(glyph)
        return glyphs
    
    raise ValueError(f"Word '{word}' not found in dictionary or morph-phoneme data.")

get_word_glyphs("necessity")

['⿰言⿱女攵', '⿰言⿱干丮']

In [16]:
import stanza
nlp = stanza.Pipeline('en')

d:\Projects\playground\kage-engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-26 22:45:35 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-10-26 22:45:36 INFO: Downloaded file to C:\Users\Peter\stanza_resources\resources.json
2025-10-26 22:45:49 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm        

In [70]:
punct_conversion = {
    ".": "。", 
    ",": "，", 
    "?": "？", 
    "!": "！", 
    ";": "；", 
    ":": "：",
}

with open("dictionary.json", "r", encoding="utf-8") as f:
    dictionary = json.load(f)

def convert_text(text: str):
    doc = nlp(text)
    converted_words = []
    for sentence in doc.sentences: # type: ignore
        for word in sentence.words:
            if word.text in punct_conversion:
                converted_words.append((word.text, word.text, [punct_conversion[word.text]]))
                continue
            lemma = word.lemma.lower()
            if lemma in ["person", "people", "be"]:
                lemma = word.text.lower()
            word_glyphs = get_word_glyphs(lemma)
            feats = word.feats if word.feats else ""
            pos = word.pos
            if pos == 'NOUN' and 'Number=Plur' in feats:
                word_glyphs.append(dictionary["s[pl]"])
            if pos == 'VERB' and 'Number=Sing' in feats and 'Person=3' in feats:
                word_glyphs.append(dictionary["s[3ps]"])
            if pos == 'VERB' and 'Tense=Past' in feats:
                if 'VerbForm=Part' in feats:
                    word_glyphs.append(dictionary["[past part]"])
                else:
                    word_glyphs.append(dictionary["ed[past]"])
            if 'Tense=Pres' in feats and 'VerbForm=Part' in feats:
                word_glyphs.append(dictionary["ing"])
            if 'Degree=Cmp' in feats:
                word_glyphs.append(dictionary["er"])
            if 'Degree=Sup' in feats:
                word_glyphs.append(dictionary["est"])
            
            
            converted_words.append((word.text, word.lemma, word_glyphs))
    return converted_words

In [72]:
text = "I am happy. He is happier. She is happiest. You eat, they ate, we are eating, you have walked, they have been running. It rains every day with mice and men."
convert_text(text)

[('I', 'I', ['⿰人目']),
 ('am', 'be', ['⿰个乙']),
 ('happy', 'happy', ['⿰心⿱古车']),
 ('.', '.', ['。']),
 ('He', 'he', ['⿰人也']),
 ('is', 'be', ['乙']),
 ('happier', 'happy', ['⿰心⿱古车', '⿰上耳']),
 ('.', '.', ['。']),
 ('She', 'she', ['⿰女也']),
 ('is', 'be', ['乙']),
 ('happiest', 'happy', ['⿰心⿱古车', '⿱上上']),
 ('.', '.', ['。']),
 ('You', 'you', ['尔']),
 ('eat', 'eat', ['食']),
 (',', ',', ['，']),
 ('they', 'they', ['⿰共也']),
 ('ate', 'eat', ['食', '了']),
 (',', ',', ['，']),
 ('we', 'we', ['共']),
 ('are', 'be', ['⿰几乙']),
 ('eating', 'eat', ['食', '者']),
 (',', ',', ['，']),
 ('you', 'you', ['尔']),
 ('have', 'have', ['有']),
 ('walked', 'walk', ['⿱止止', '矣']),
 (',', ',', ['，']),
 ('they', 'they', ['⿰共也']),
 ('have', 'have', ['有']),
 ('been', 'be', ['也', '矣']),
 ('running', 'run', ['⿱犬田', '者']),
 ('.', '.', ['。']),
 ('It', 'it', ['⿰乙也']),
 ('rains', 'rain', ['雨', '乙']),
 ('every', 'every', ['⿰个⿱车舟']),
 ('day', 'day', ['⿱日天']),
 ('with', 'with', ['⿰上⿱木心']),
 ('mice', 'mouse', ['⿰犬⿱彡虫', '几']),
 ('and', 'and', ['